# Notebook 03: Scenario Planning Dashboard
**Data Sources:**
- BLS LAUS — 72 monthly labor market records (2019–2024)
- Census ACS — 1 demographic snapshot (2023)
- DC Agency Metrics — 10 agency performance indices (2024-Q1)

**Method:** What-if budget reallocation using historical correlation + scenario modeling.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from pathlib import Path

BASE = Path('/root/.openclaw/workspace/sierra-pmo-analytics/projects/executive-decision-support')

# Load data
bls = pd.read_csv(BASE / 'data/bls_dc_wide.csv')
bls['date'] = pd.to_datetime(bls['year'].astype(str) + '-' + bls['period'].str.replace('M', '') + '-01')
census = pd.read_csv(BASE / 'data/census_dc.csv')
agencies = pd.read_csv(BASE / 'data/dc_agency_metrics.csv')

HISTORICAL_SHARES = {
    "Department of Health": 0.08,
    "Metropolitan Police Department": 0.18,
    "Department of Transportation": 0.07,
    "Department of Energy & Environment": 0.04,
    "Office of the Chief Technology Officer": 0.03,
    "Department of Human Services": 0.12,
    "Department of Parks and Recreation": 0.05,
    "DC Public Schools": 0.28,
    "Department of Employment Services": 0.06,
    "Department of Consumer and Regulatory Affairs": 0.04,
}
TOTAL_BUDGET = 20.6e9
print('Data loaded. Ready for scenario modeling.')

Data loaded. Ready for scenario modeling.


## Chart 1: Scenario Comparison — Baseline vs Optimistic vs Pessimistic
**Method:** Baseline = historical shares. Optimistic = boost high-performing agencies (+15% to top 4). Pessimistic = cut all shares 10% and reallocate to safety net.
**Assumption:** 30% performance elasticity to budget change (heuristic).

In [2]:
def run_scenario(allocations):
    results = []
    for _, row in agencies.iterrows():
        agency = row['agency_name']
        base_perf = row['value']
        base_share = HISTORICAL_SHARES.get(agency, 0.05)
        share = allocations.get(agency, base_share)
        change = (share - base_share) / base_share if base_share > 0 else 0
        projected = base_perf * (1 + 0.3 * change)
        results.append({
            'Agency': agency,
            'Budget Share': share,
            'Budget ($)': share * TOTAL_BUDGET,
            'Projected Performance': round(min(100, projected), 1),
            'Change from Baseline': round(projected - base_perf, 1)
        })
    return pd.DataFrame(results)

# Build scenarios
baseline = HISTORICAL_SHARES.copy()

# Optimistic: boost top 4 performers by 15%, normalize
optimistic = baseline.copy()
top4 = agencies.nlargest(4, 'value')['agency_name'].tolist()
for a in top4:
    optimistic[a] *= 1.15
total = sum(optimistic.values())
optimistic = {k: v/total for k, v in optimistic.items()}

# Pessimistic: cut all 10%, boost safety net (Human Services, Employment Services, Health) by 20%
pessimistic = {k: v * 0.90 for k, v in baseline.items()}
for a in ['Department of Human Services', 'Department of Employment Services', 'Department of Health']:
    pessimistic[a] *= 1.20
total = sum(pessimistic.values())
pessimistic = {k: v/total for k, v in pessimistic.items()}

df_base = run_scenario(baseline)
df_opt = run_scenario(optimistic)
df_pes = run_scenario(pessimistic)

# Merge for comparison
comp = df_base[['Agency', 'Projected Performance']].rename(columns={'Projected Performance': 'Baseline'})
comp = comp.merge(df_opt[['Agency', 'Projected Performance']].rename(columns={'Projected Performance': 'Optimistic'}), on='Agency')
comp = comp.merge(df_pes[['Agency', 'Projected Performance']].rename(columns={'Projected Performance': 'Pessimistic'}), on='Agency')

fig = go.Figure()
for col, color in [('Baseline', 'steelblue'), ('Optimistic', 'darkgreen'), ('Pessimistic', 'firebrick')]:
    fig.add_trace(go.Bar(name=col, x=comp['Agency'], y=comp[col], marker_color=color))
fig.update_layout(
    barmode='group',
    title='Scenario Comparison: Projected Agency Performance by Budget Scenario',
    xaxis_tickangle=-45,
    yaxis_title='Projected Performance Index',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
)
fig.write_image(str(BASE / 'figures/14_scenario_comparison.png'), scale=2)
fig.show()

/tmp/ipykernel_120827/4223435735.py:57: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(str(BASE / 'figures/14_scenario_comparison.png'), scale=2)


## Chart 2: Interactive Budget Allocation Waterfall
Source: Scenario modeling — Shows the delta from baseline to optimistic scenario per agency

In [3]:
delta = df_opt[['Agency', 'Change from Baseline']].copy()
delta['Direction'] = delta['Change from Baseline'].apply(lambda x: 'Increase' if x > 0 else 'Decrease')

fig = px.bar(
    delta,
    x='Agency', y='Change from Baseline', color='Direction',
    color_discrete_map={'Increase': 'darkgreen', 'Decrease': 'firebrick'},
    title='Optimistic Scenario: Performance Delta from Baseline',
    labels={'Change from Baseline': 'Δ Performance Index'}
)
fig.update_layout(xaxis_tickangle=-45, height=450)
fig.write_image(str(BASE / 'figures/15_delta_waterfall.png'), scale=2)
fig.show()

/tmp/ipykernel_120827/187915917.py:12: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(str(BASE / 'figures/15_delta_waterfall.png'), scale=2)


## Chart 3: Executive KPI Cards — Labor + Demographic + Scenario
Interactive KPI summary combining all three data sources

In [4]:
latest_ur = bls.sort_values('date').iloc[-1]['dc_unemployment_rate']
latest_emp = bls.sort_values('date').iloc[-1]['dc_employment_level']
pop = census['population'].iloc[0]
inc = census['median_income'].iloc[0]
avg_perf = agencies['value'].mean()
opt_avg = df_opt['Projected Performance'].mean()
pes_avg = df_pes['Projected Performance'].mean()

fig = make_subplots(
    rows=2, cols=3,
    specs=[[{'type': 'indicator'}, {'type': 'indicator'}, {'type': 'indicator'}],
           [{'type': 'indicator'}, {'type': 'indicator'}, {'type': 'indicator'}]],
    vertical_spacing=0.25
)

fig.add_trace(go.Indicator(
    mode='number+delta',
    value=latest_ur,
    title={'text': 'Unemployment Rate (Dec 2024)'},
    delta={'reference': 5.5, 'relative': False, 'valueformat': '.1f', 'suffix': '%'},
    number={'suffix': '%', 'font': {'size': 36}},
), row=1, col=1)

fig.add_trace(go.Indicator(
    mode='number',
    value=latest_emp,
    title={'text': 'Employment Level (thousands)'},
    number={'font': {'size': 36}},
), row=1, col=2)

fig.add_trace(go.Indicator(
    mode='number',
    value=pop,
    title={'text': 'DC Population (ACS 2023)'},
    number={'valueformat': ',', 'font': {'size': 32}},
), row=1, col=3)

fig.add_trace(go.Indicator(
    mode='number',
    value=inc,
    title={'text': 'Median Income ($)'},
    number={'prefix': '$', 'valueformat': ',', 'font': {'size': 32}},
), row=2, col=1)

fig.add_trace(go.Indicator(
    mode='gauge+number',
    value=avg_perf,
    title={'text': 'Avg Agency Performance'},
    gauge={'axis': {'range': [0, 100]}, 'bar': {'color': 'steelblue'},
           'steps': [{'range': [0, 60], 'color': '#ffcccc'},
                     {'range': [60, 80], 'color': '#ffffcc'},
                     {'range': [80, 100], 'color': '#ccffcc'}]},
), row=2, col=2)

fig.add_trace(go.Indicator(
    mode='number+delta',
    value=opt_avg,
    title={'text': 'Optimistic Scenario Avg'},
    delta={'reference': avg_perf, 'relative': False, 'valueformat': '.1f'},
    number={'font': {'size': 36}},
), row=2, col=3)

fig.update_layout(height=600, title_text='Executive KPI Dashboard — DC Strategic Indicators', title_x=0.5)
fig.write_image(str(BASE / 'figures/16_executive_kpis.png'), scale=2)
fig.show()

/tmp/ipykernel_120827/2688446950.py:64: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(str(BASE / 'figures/16_executive_kpis.png'), scale=2)


## Chart 4: Sensitivity Analysis — MPD Budget ±10%
Source: Scenario engine — Shows nonlinear sensitivity of performance to budget shifts for Metropolitan Police Department

In [5]:
agency = 'Metropolitan Police Department'
base_share = HISTORICAL_SHARES[agency]
base_perf = agencies[agencies['agency_name']==agency]['value'].values[0]

sens = []
for pct in np.linspace(-0.20, 0.20, 9):
    new_share = base_share * (1 + pct)
    change = (new_share - base_share) / base_share
    new_perf = base_perf * (1 + 0.3 * change)
    sens.append({'Shift': f"{pct*100:+.0f}%", 'Share': new_share, 'Performance': round(min(100, new_perf), 1)})

sens_df = pd.DataFrame(sens)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=sens_df['Shift'], y=sens_df['Performance'],
    mode='lines+markers',
    line=dict(color='steelblue', width=3),
    marker=dict(size=10),
    fill='tozeroy',
    fillcolor='rgba(70,130,180,0.2)'
))
fig.add_hline(y=base_perf, line_dash='dash', annotation_text=f'Baseline = {base_perf}')
fig.update_layout(
    title='Sensitivity: MPD Performance vs Budget Shift (±20%)',
    xaxis_title='Budget Shift from Historical Baseline',
    yaxis_title='Projected Performance Index',
    height=400
)
fig.write_image(str(BASE / 'figures/17_sensitivity_mpd.png'), scale=2)
fig.show()

/tmp/ipykernel_120827/2317335821.py:30: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(str(BASE / 'figures/17_sensitivity_mpd.png'), scale=2)


## Scenario Summary Export
All three scenarios saved for dashboard integration.

In [6]:
df_base.to_json(BASE / 'data/scenario_baseline.json', orient='records', indent=2)
df_opt.to_json(BASE / 'data/scenario_optimistic.json', orient='records', indent=2)
df_pes.to_json(BASE / 'data/scenario_pessimistic.json', orient='records', indent=2)
print('Scenarios exported to data/')
print(f"  Baseline avg performance: {df_base['Projected Performance'].mean():.1f}")
print(f"  Optimistic avg performance: {df_opt['Projected Performance'].mean():.1f}")
print(f"  Pessimistic avg performance: {df_pes['Projected Performance'].mean():.1f}")

Scenarios exported to data/
  Baseline avg performance: 69.1
  Optimistic avg performance: 70.6
  Pessimistic avg performance: 70.2
